In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
from birddog.database import Database
from birddog.wiki import (
    page_label,
    sequential_page_label,
)

2026-02-18 14:25:18,681 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com
2026-02-18 14:25:18,834 [INFO] Translation is enabled. Using GCP translator
2026-02-18 14:25:18,835 [INFO] Using Google Cloud translation API
2026-02-18 14:25:18,835 [INFO] GoogleCloudTranslator using REST API


In [3]:
db = Database()

2026-02-18 14:25:19,179 [INFO] 
AdaptiveThrottle report:
  host_key                                rps   tokens  blocked_s max_in_flight
  ------------------------------------------------------------------------------
  nocodb.internal:api                   20.00    19.00       0.00           16


In [ ]:
pages = []
cursor = None
while True:
    print(f"cursor={cursor}")
    batch, cursor = db.scan(
        "Pages", 
        cursor=cursor, 
        limit=10000,
        where=("source_type", "neq", "wiki"),
        fields="Id")
    if batch:
        pages.extend([b["Id"] for b in batch])
    if not cursor:
        break

In [ ]:
len(pages)

In [ ]:
#db.delete("Pages", pages)

In [ ]:
docs = []
cursor = None
while True:
    print(f"cursor={cursor}")
    batch, cursor = db.scan(
        "Documents", 
        cursor=cursor, 
        limit=1000,
        where=("owning_pages", "eq", 0),
        fields="Id",
    )
    if batch:
        docs.extend([b["Id"] for b in batch])
    if not cursor:
        break
    #break

In [ ]:
docs[0]

In [ ]:
len(docs)

In [ ]:
#db.delete("Documents", docs)

In [26]:
pages = []
cursor = None
while True:
    if cursor and (int(cursor) % 10000) == 0:
        print(f"cursor={cursor}")
    batch, cursor = db.scan(
        "Pages", 
        cursor=cursor, 
        limit=1000,
        where=("seq_label", "is", None),
        fields=("Id","title","label","seq_label"),
    )
    if batch:
        pages.extend(batch)
    if not cursor:
        break

2026-02-18 14:37:24,223 [INFO] 
AdaptiveThrottle report:
  host_key                                rps   tokens  blocked_s max_in_flight
  ------------------------------------------------------------------------------
  nocodb.internal:api                   26.00    19.00       0.00           16
cursor=10000
cursor=20000
cursor=30000
cursor=40000
cursor=50000
cursor=60000
cursor=70000
cursor=80000
cursor=90000
cursor=100000
cursor=110000
cursor=120000
cursor=130000
cursor=140000
cursor=150000


In [27]:
len(pages)

155328

In [28]:
pages[1000]

{'Id': 7616, 'title': 'ДАВоО/Р-3247/2/572', 'label': None, 'seq_label': None}

In [29]:
def normalize_labels(page):
    result = page.copy()
    title = page.get("title")
    if title:
        proper_label = page_label(title)
        result["label"] = proper_label
        proper_seq_label = sequential_page_label(proper_label)
        result["seq_label"] = proper_seq_label
    return result

In [30]:
normalize_labels(pages[2000])

{'Id': 8715,
 'title': 'ДАЖО/1/74',
 'label': 'DAZHO-D/1/74',
 'seq_label': 'DAZHO-D/00000001/00000074'}

In [31]:
pages[2000]

{'Id': 8715, 'title': 'ДАЖО/1/74', 'label': 'DAZHO-D/1/74', 'seq_label': None}

In [32]:
pages[2000] == normalize_labels(pages[2000])

False

In [33]:
pages[2000] == pages[2000].copy()

True

In [34]:
norm_pages = [normalize_labels(p) for p in pages]

In [35]:
norm_pages[:10]

[{'Id': 6599,
  'title': 'ДАВоО/35/9/318',
  'label': 'DAVO-D/35/9/318',
  'seq_label': 'DAVO-D/00000035/00000009/00000318'},
 {'Id': 6600,
  'title': 'ДАВоО/35/9',
  'label': 'DAVO-D/35/9',
  'seq_label': 'DAVO-D/00000035/00000009'},
 {'Id': 6601,
  'title': 'ДАЧкО/8/2/330',
  'label': 'DACHKO-D/8/2/330',
  'seq_label': 'DACHKO-D/00000008/00000002/00000330'},
 {'Id': 6602,
  'title': 'ДАЧкО/8/2',
  'label': 'DACHKO-D/8/2',
  'seq_label': 'DACHKO-D/00000008/00000002'},
 {'Id': 6603,
  'title': 'ДАДнО/Р-6508/7/244',
  'label': 'DADNO-R/R-6508/7/244',
  'seq_label': 'DADNO-R/R-00006508/00000007/00000244'},
 {'Id': 6604,
  'title': 'ДАВіО/904/31',
  'label': 'DAVIO-D/904/31',
  'seq_label': 'DAVIO-D/00000904/00000031'},
 {'Id': 6605,
  'title': 'ДАКО/280/71',
  'label': 'DAKO-D/280/71',
  'seq_label': 'DAKO-D/00000280/00000071'},
 {'Id': 6606,
  'title': 'ДАДнО/Р-6508/3/144',
  'label': 'DADNO-R/R-6508/3/144',
  'seq_label': 'DADNO-R/R-00006508/00000003/00000144'},
 {'Id': 6607,
  'title'

In [36]:
changed_pages = [n for n,p in zip(norm_pages, pages) if n != p]

In [37]:
len(changed_pages)

155328

In [38]:
len(pages)

155328

In [39]:
len(norm_pages)

155328

In [42]:
rec_ids = db.write("Pages", norm_pages[1000:2000])

2026-02-18 14:45:18,289 [INFO] 
AdaptiveThrottle report:
  host_key                                rps   tokens  blocked_s max_in_flight
  ------------------------------------------------------------------------------
  nocodb.internal:api                   37.00    19.00       0.00           16


In [44]:
chunk = 1000
for i in range(0, len(norm_pages), chunk):
    print(i)
    rec_ids = db.write("Pages", norm_pages[i:(i+chunk)])

0
2026-02-18 14:47:19,710 [INFO] 
AdaptiveThrottle report:
  host_key                                rps   tokens  blocked_s max_in_flight
  ------------------------------------------------------------------------------
  nocodb.internal:api                   42.00    19.00       0.00           16
1000
2026-02-18 14:48:19,936 [INFO] 
AdaptiveThrottle report:
  host_key                                rps   tokens  blocked_s max_in_flight
  ------------------------------------------------------------------------------
  nocodb.internal:api                   47.00    19.00       0.00           16
2000
2026-02-18 14:49:20,060 [INFO] 
AdaptiveThrottle report:
  host_key                                rps   tokens  blocked_s max_in_flight
  ------------------------------------------------------------------------------
  nocodb.internal:api                   53.00    19.00       0.00           16
3000
4000
2026-02-18 14:50:20,096 [INFO] 
AdaptiveThrottle report:
  host_key                    